# Requested Analytic Transformation

Analytics tables built for business teams from the enriched transactions and the customer table.

**Reads:** `silver.silver_transaction_enriched`, `silver.silver_customers`  
**Writes:** `gold.gold_customer_aggregate`, `gold.gold_top_n_category_customer`, `gold.gold_customer_monthly_spend`, `gold.gold_customer_rfm_segmentation`, `gold.gold_top_5_merchants_of_city` and `gold.gold_channel_category_spending_summary`

## Imports

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Shared functions

In [ ]:
%run NB_FUNCTIONS

## Load source tables

In [ ]:
# Importing txn_fx enriched DataFrame
txn_fx = spark.read.table("silver.silver_transaction_enriched")

# importing silver_customer DataFrame
cust_df = spark.read.table("silver.silver_customers")

## Aggregated Per Customer
- Relationship managers want following attributes in one row per customer:
    1) spend,
    2) income,
    3) number of transactions,
    4) average ticket,
    5) first and last activity
    6) the number of alerts.

In [ ]:
success_ok = txn_fx.filter("status = 'SUCCESS'")

cust_metrics = success_ok.groupBy("customer_id").agg(
    F.sum(F.when(F.col("txn_type") == "DEBIT", F.col("amount_inr"))).alias("spend_inr"),
    F.sum(F.when(F.col("txn_type") != "DEBIT", F.col("amount_inr"))).alias(
        "inflow_inr"
    ),
    F.count("*").alias("txn_count"),
    F.round(F.avg("amount_inr"), 2).alias("avg_ticket_inr"),
    F.min("txn_ts").alias("first_txn_ts"),
    F.max("txn_ts").alias("last_txn_ts"),
    F.countDistinct("merchant_merchant_id").alias("distinct_merchants"),
    F.sum(F.col("risk_band").isin("HIGH", "MEDIUM").cast("int")).alias("alerts"),
)


customer_agg = (
    cust_df.select(
        "customer_id",
        "full_name",
        "city",
        "age",
        "income_band",
        "kyc_status",
        "risk_rating",
    )
    .join(cust_metrics, "customer_id", "left")
    .fillna({"txn_count": 0, "alerts": 0})
)


customer_agg.show()

In [ ]:
# routing it to gold layer
delta_merge(
    df=customer_agg,
    schema="gold",
    table_name="gold_customer_aggregate",
    merge_cond="target.customer_id = source.customer_id",
)

## Top N category per customer
- Find Customers with Top N category with the highest spend.

In [ ]:
catg_spend = (
    success_ok.filter("txn_type = 'DEBIT'")
    .groupBy("customer_id", "category")
    .agg(F.sum("amount_inr").alias("cat_spend_inr"))
)

# window
w_cat = Window.partitionBy("customer_id").orderBy(F.desc("cat_spend_inr"), "category")

fav = (
    catg_spend.withColumn("rk", F.row_number().over(w_cat))
    .filter("rk = 1")
    .select("customer_id", F.col("category").alias("favourite_category"))
)

top_n_category_customer = customer_agg.join(fav, "customer_id", "left")

top_n_category_customer.show()

In [ ]:
# routing it to gold layer
delta_merge(
    df=top_n_category_customer,
    schema="gold",
    table_name="gold_top_n_category_customer",
    merge_cond="target.customer_id = source.customer_id",
)

## Monthly Spend Matrix
- Finance wants one row per customer and one column per month

In [ ]:
MONTHS = ["2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"]
monthly_customer_spend = (
    success_ok.filter("txn_type = 'DEBIT'")
    .withColumn("month", F.date_format("txn_date_ist", "yyyy-MM"))
    .groupBy("customer_id")
    .pivot("month", MONTHS)
    .agg(F.round(F.sum("amount_inr"), 2))
    .fillna(0)
)
monthly_customer_spend.show()

In [ ]:
# routing it to gold layer
delta_merge(
    df=monthly_customer_spend,
    schema="gold",
    table_name="gold_customer_monthly_spend",
    merge_cond="target.customer_id = source.customer_id",
)

## RFM Customer Segmentation Model

**Requester:** Marketing Team

Classify customers based on their transaction behavior using three dimensions:
- R -- **Recency**: How recently did the customer transact?
- F -- **Frequency**: How often does the customer transact?
- M -- **Monetary**: How much money does the customer spend?


In [ ]:
AS_OF = "2025-07-01"
rfm = (
    customer_agg.filter("txn_count > 0")
    .withColumn("recency_days", F.datediff(F.lit(AS_OF), F.to_date("last_txn_ts")))
    .withColumn("r", 6 - F.ntile(5).over(Window.orderBy("recency_days")))
    .withColumn("f", F.ntile(5).over(Window.orderBy("txn_count")))
    .withColumn(
        "m", F.ntile(5).over(Window.orderBy(F.col("spend_inr").asc_nulls_first()))
    )
    .withColumn(
        "segment",
        F.when((F.col("r") >= 4) & (F.col("f") >= 4), "CHAMPION")
        .when(F.col("r") <= 2, "AT_RISK")
        .when(F.col("m") >= 4, "BIG_SPENDER")
        .otherwise("REGULAR"),
    )
)

rfm.show()

In [ ]:
# routing it to gold layer
delta_merge(
    df=rfm,
    schema="gold",
    table_name="gold_customer_rfm_segmentation",
    merge_cond="target.customer_id = source.customer_id",
)

## Top 5 Merchants Per City

**Requester:** Partnership Team


In [ ]:
merchant_spend = (
    success_ok.filter("merchant_merchant_id IS NOT NULL AND txn_type = 'DEBIT'")
    .groupBy("merchant_location_city", "merchant_name")
    .agg(F.round(F.sum("amount_inr"), 0).alias("spend_inr"), F.count("*").alias("txns"))
)

# window
w_city = Window.partitionBy("merchant_location_city").orderBy(F.desc("spend_inr"))

top_5_merchants = (
    merchant_spend.withColumn("rank", F.dense_rank().over(w_city))
    .filter("rank <= 5")
    .orderBy("merchant_location_city", "rank")
)

top_5_merchants.show(25, truncate=False)

In [ ]:
# routing it to gold layer
delta_merge(
    df=top_5_merchants,
    schema="gold",
    table_name="gold_top_5_merchants_of_city",
    merge_cond="target.merchant_location_city = source.merchant_location_city AND target.merchant_name = source.merchant_name",
)

## Total & Sub-Total of Payment Channel and Category

The CFO's table: spend by channel and category, with a subtotal per channel and a grand total.

In [ ]:
summary = (
    success_ok.filter("txn_type = 'DEBIT'")
    .rollup("channel", "category")
    .agg(
        F.round(F.sum("amount_inr") / 1e7, 2).alias("spend_crore"),
        F.count("*").alias("txns"),
    )
    .withColumn(
        "level",
        F.when(F.col("channel").isNull(), "GRAND TOTAL")
        .when(F.col("category").isNull(), "CHANNEL TOTAL")
        .otherwise("DETAIL"),
    )
    .orderBy(F.col("channel").asc_nulls_last(), F.col("category").asc_nulls_last())
)

summary.show(50, truncate=False)

In [ ]:
# routing it to gold layer
delta_merge(
    df=summary,
    schema="gold",
    table_name="gold_channel_category_spending_summary",
    merge_cond="target.channel = source.channel AND target.category = source.category",
)